In [6]:
import requests
import pathlib
import xml.etree.ElementTree as ET
import json
import time
import string
import sys
import pandas as pd
from deutsche_bahn_api import ApiAuthentication
from pathlib import Path
import os
import numpy as np
import re
from datetime import datetime, timedelta
from math import radians, cos, sin, asin, sqrt


BASE_DB_API = "https://apis.deutschebahn.com/db-api-marketplace/apis/"
STOP_PLACES_URL = "https://apis.deutschebahn.com/db-api-marketplace/apis/ris-stations/v1/stop-places"
TIMETABLES_V1_URL = BASE_DB_API + "timetables/v1"
RIS_STATION_URL = BASE_DB_API + "ris-station/v1/stations/"
ROOT_DIR = Path(os.getcwd())
DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = ROOT_DIR / "Raw_Data"

DB_CLIENT_ID = "a9f83c55d26c3ee7f48f4ce887ec2a57"
DB_API_KEY = "422cac21a0a83876c75efb8806589ea0"
PKP_API_KEY = "KMjLifnR-a6RGlVgs36-OGG82nZ2gZRuiySF_y-tWSbUMX4LNS3JfBk1hli1B59YIXfdIrIy3ZvwUpkrMueAeA"

header = {
    "DB-Client-Id": DB_CLIENT_ID,
    "DB-Api-Key": DB_API_KEY,
    "accept": "application/xml"
}

header2 = {
    "DB-Client-Id": DB_CLIENT_ID,
    "DB-Api-Key": DB_API_KEY,
    "accept": "application/json"
}

def get_all_stations(country_code="DE"):
    params = {"state": country_code}
    
    response = requests.get(RIS_STATION_URL, headers=header, params=params)
    if response.status_code == 200:
        return response.json().get('stations', [])
    else:   
        print(f"Error: {response.status_code}")
        return []

def haversine(lat1, lon1, lat2, lon2):
    """ Calculate distance in km between two points """
    R = 6371 # Earth radius
    dLat = radians(lat2 - lat1)
    dLon = radians(lon2 - lon1)
    a = sin(dLat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dLon/2)**2
    return 2 * R * asin(sqrt(a))

# api credentials validation
api_authentication = ApiAuthentication( DB_CLIENT_ID, DB_API_KEY)
success:bool = api_authentication.test_credentials()
success

True

#  Railway stations selection & filtering

First, match train station with selected cities from simplemaps ( Processed_Data/cities_500K). Later for matched cities retrieve required timetables.

Matching should be done geospatially

In [7]:
PROCESSED_DATA_DIR = ROOT_DIR / "Processed_Data"

cities_data = pd.read_csv(PROCESSED_DATA_DIR / "cities_500K.csv")
cities_data.rename(columns={"name_city":"name", "lat_city":"latitude", "lng_city":"longitude","country":"iso_code"}, inplace=True)
cities_data.head()

,name,latitude,longitude,iso_code,population,is_capital
0,Vienna,48.2083,16.3725,AT,1973403.0,True
1,Brussels,50.8467,4.3525,BE,1235192.0,True
2,Antwerp,51.2178,4.4003,BE,536079.0,False
3,Sofia,42.7000,23.3300,BG,1383435.0,True
4,Prague,50.0875,14.4214,CZ,1357326.0,True


In [8]:
RIS_STATION_URL = "https://apis.deutschebahn.com/db-api-marketplace/apis/ris-stations/v1/stations"
header_ris = {
    "DB-Client-Id": DB_CLIENT_ID,
    "DB-Api-Key": DB_API_KEY,
    "accept": "application/vnd.de.db.ris+json"
}

res = requests.get(url=RIS_STATION_URL, headers=header_ris, params={"countryCode":"AT"})
res.text

'{"offset":0,"limit":100,"total":5703,"stations":[{"stationID":"1","names":{"DE":{"name":"Aachen Hbf"}},"metropolis":{},"address":{"street":"Bahnhofstr.","houseNumber":"2a","postalCode":"52064","city":"Aachen","state":"Nordrhein-Westfalen","country":"DE"},"stationCategory":"CATEGORY_2","availableTransports":[],"availableLocalServices":[],"transportAssociations":[],"owner":{"name":"DB InfraGO Personenbahnhöfe","organisationalUnit":{"id":4,"name":"RB West","nameShort":"RB West"}},"countryCode":"DE","state":"NW","timeZone":"Europe/Berlin","position":{"longitude":6.091499,"latitude":50.7678},"validFrom":"2018-12-31T23:00:00Z","mobilityServiceStaffOnSite":true},{"stationID":"1000","names":{"DE":{"name":"Burkhardswalde-Maxen"}},"metropolis":{},"address":{"street":"Gesundbrunnen","houseNumber":"60c","postalCode":"01809","city":"Müglitztal-Burkhardswalde","state":"Sachsen","country":"DE"},"stationCategory":"CATEGORY_7","availableTransports":[],"availableLocalServices":[],"transportAssociations

Testing stop-places accesspoint

In [10]:
response = requests.get(url=f"{STOP_PLACES_URL}/by-name/Warsaw?sortBy=RELEVANCE&onlyActive=true&withSynonyms=true&limit=3", headers=header_ris)
response.text

'{"stopPlaces":[{"evaNumber":"5100065","groupMembers":[],"names":{"DE":{"nameLong":"Warszawa Centralna","synonyms":[]}},"replacementTransportsAvailable":false,"availableTransports":["INTERCITY_TRAIN","INTER_REGIONAL_TRAIN"],"position":{"longitude":21.003234,"latitude":52.22886}},{"evaNumber":"5100067","groupMembers":[],"names":{"DE":{"nameLong":"Warszawa Zachodnia","synonyms":[]}},"replacementTransportsAvailable":false,"availableTransports":["INTERCITY_TRAIN","INTER_REGIONAL_TRAIN"],"position":{"longitude":20.965247,"latitude":52.219972}},{"evaNumber":"5100066","groupMembers":[],"names":{"DE":{"nameLong":"Warszawa Wschodnia","synonyms":[]}},"replacementTransportsAvailable":false,"availableTransports":["INTERCITY_TRAIN","INTER_REGIONAL_TRAIN"],"position":{"longitude":21.052335,"latitude":52.251548}}]}'

Using RIS:stations retrieve all cities main stations

In [17]:
def get_stations_api_stop_places(cities : pd.DataFrame, 
                      base_url: str = STOP_PLACES_URL,  
                      limit: int = 3) -> pd.DataFrame:
    """
    Retrieves station information from the Deutsche Bahn API based on the station name.
    
    Keyword arguments:
    cities -- DataFrame containing city information
    base_url -- Base URL for the API
    limit -- Maximum number of stations to retrieve per city
    Return: DataFrame of stations for given cities
    """
    retrieved_stations = pd.DataFrame(columns=['city_name', 'eva_id', 'station_name', 'latitude', 'longitude'])
    for _, row in cities.iterrows():
        city_name = row['name']
        response = None
        params = {
            "sortBy": "RELEVANCE",
            "onlyActive": "true",
            "withSynonyms": "true",
            "latitude": row['latitude'],
            "longitude": row['longitude'],
            "limit": limit
        }
        success = False
        retries = 0

        while not success and retries < 3:
            response = requests.get(url=f"{base_url}/by-name/{city_name}", params=params, headers=header_ris)

            if response.status_code == 429:
                print(f"Rate limit reached. Sleeping for 1s...")
                time.sleep(1)
                retries += 1
                continue

            if response.status_code == 200:
                stations = response.json().get('stopPlaces', [])
                print(f"City: {city_name}, Stations Found: {len(stations)}")
                for station in stations:
                    latitude = float(station.get('position').get('latitude'))
                    longitude = float(station.get('position').get('longitude'))

                    if haversine(row['latitude'], row['longitude'], latitude, longitude) > 10:
                        print(f"Skipping station {station.get('names').get('DE').get('nameLong')} due to distance.")
                        continue

                    retrieved_stations = pd.concat([retrieved_stations, pd.DataFrame({
                        'city_name': [city_name],
                        'eva_id': [str(station.get('evaNumber'))],
                        'station_name': [str(station.get('names').get('DE').get('nameLong'))],
                        'latitude': [float(station.get('position').get('latitude'))],
                        'longitude': [float(station.get('position').get('longitude'))]
                    })])
                success = True
            else:
                print(f"Error retrieving stations for city {city_name}: {response.status_code}")
                break

            time.sleep(0.09)  # To respect API rate limits

    return retrieved_stations

In [12]:
test_cities = cities_data.iloc[np.random.choice(cities_data.shape[0], 5, replace=False)]
test_cities

,name,latitude,longitude,iso_code,population,is_capital
14,Essen,51.4508,7.0131,DE,584580.0,False
55,Bucharest,44.4325,26.1039,RO,1716961.0,True
54,Lisbon,38.7253,-9.1500,PT,548703.0,True
31,Marseille,43.2964,5.3700,FR,873076.0,False
19,Duisburg,51.4347,6.7625,DE,502211.0,False


In [96]:
output  = get_stations_api_stop_places(test_cities, limit=3)

City: Málaga, Stations Found: 0
City: Essen, Stations Found: 3


C:\Users\Adam\AppData\Local\Temp\ipykernel_15984\3773918198.py:48: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  retrieved_stations = pd.concat([retrieved_stations, pd.DataFrame({


City: Warsaw, Stations Found: 3
City: Paris, Stations Found: 3
City: Vilnius, Stations Found: 1
Skipping station Vilniuser Straße, Erfurt due to distance.


For cities above 500K we select at least 3 station if possible as timetables for those stations might contain most crutial connections

In [18]:
citiies_stations = get_stations_api_stop_places(cities_data, limit=1)

City: Vienna, Stations Found: 1


C:\Users\Adam\AppData\Local\Temp\ipykernel_3492\580966334.py:48: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  retrieved_stations = pd.concat([retrieved_stations, pd.DataFrame({


City: Brussels, Stations Found: 1
City: Antwerp, Stations Found: 1
City: Sofia, Stations Found: 1
Skipping station Sofie-Hammer-Straße, Osnabrück due to distance.
City: Prague, Stations Found: 1
City: Berlin, Stations Found: 1
City: Stuttgart, Stations Found: 1
City: Munich, Stations Found: 1
City: Hamburg, Stations Found: 1
City: Cologne, Stations Found: 1
City: Frankfurt, Stations Found: 1
City: Düsseldorf, Stations Found: 1
City: Leipzig, Stations Found: 1
City: Dortmund, Stations Found: 1
City: Essen, Stations Found: 1
City: Bremen, Stations Found: 1
City: Dresden, Stations Found: 1
City: Hannover, Stations Found: 1
City: Nuremberg, Stations Found: 1
City: Duisburg, Stations Found: 1
City: Copenhagen, Stations Found: 1
City: Tallinn, Stations Found: 1
Skipping station Tallinner Straße, Schwerin (Meckl) due to distance.
City: Madrid, Stations Found: 1
Skipping station Madrider Ring, Würzburg due to distance.
City: Barcelona, Stations Found: 1
City: Valencia, Stations Found: 1
Skippi

In [19]:
citiies_stations

,city_name,eva_id,station_name,latitude,longitude
0,Vienna,8103000,Wien Hbf,48.185101,16.377113
0,Brussels,8800004,Bruxelles Midi,50.835376,4.335694
0,Antwerp,8800007,Antwerpen Centraal,51.215811,4.421168
0,Prague,5400014,Praha hl.n.,50.083062,14.436039
0,Berlin,8011160,Berlin Hbf,52.525592,13.369545
0,Stuttgart,8000096,Stuttgart Hbf,48.784780,9.182757
0,Munich,8000261,München Hbf,48.140232,11.558335
0,Hamburg,8002549,Hamburg Hbf,53.552736,10.006909
0,Cologne,8003368,Köln Messe/Deutz,50.940874,6.975001
0,Frankfurt,8000105,Frankfurt(Main)Hbf,50.106682,8.662828


In [130]:
citiies_stations[citiies_stations['city_name'] == "Prague" ]

,city_name,eva_id,station_name,latitude,longitude
0,Prague,5400014,Praha hl.n.,50.083062,14.436039


Found Train Stations for given cities

first cities which had no stations at DB API

In [148]:
cities_no_station = pd.merge(cities_data, citiies_stations, how='outer',left_on=['name'], right_on=['city_name'], indicator=True).query('_merge == "left_only"')
cities_no_station.reset_index(inplace=True)
cities_no_station

,index,name,latitude_x,longitude_x,iso_code,population,is_capital,city_name,eva_id,station_name,latitude_y,longitude_y,_merge
0,6,Athens,37.9842,23.7281,GR,643452.0,True,NaN,NaN,NaN,NaN,NaN,left_only
1,24,Bucharest,44.4325,26.1039,RO,1716961.0,True,NaN,NaN,NaN,NaN,NaN,left_only
2,49,Gothenburg,57.7075,11.9675,SE,607882.0,False,NaN,NaN,NaN,NaN,NaN,left_only
3,56,Helsinki,60.1708,24.9375,FI,664921.0,True,NaN,NaN,NaN,NaN,NaN,left_only
4,62,Lisbon,38.7253,-9.1500,PT,548703.0,True,NaN,NaN,NaN,NaN,NaN,left_only
5,71,Madrid,40.4169,-3.7033,ES,3266126.0,True,NaN,NaN,NaN,NaN,NaN,left_only
6,79,Málaga,36.7194,-4.4200,ES,586384.0,False,NaN,NaN,NaN,NaN,NaN,left_only
7,80,Naples,40.8333,14.2500,IT,913462.0,False,NaN,NaN,NaN,NaN,NaN,left_only
8,89,Riga,56.9489,24.1064,LV,660187.0,True,NaN,NaN,NaN,NaN,NaN,left_only
9,94,Sevilla,37.3900,-5.9900,ES,684025.0,False,NaN,NaN,NaN,NaN,NaN,left_only


In [146]:
cities_with_stations = pd.merge(cities_data, citiies_stations, how='inner',left_on=['name'], right_on=['city_name'])
cities_with_stations

,name,latitude_x,longitude_x,iso_code,population,is_capital,city_name,eva_id,station_name,latitude_y,longitude_y
0,Vienna,48.2083,16.3725,AT,1973403.0,True,Vienna,8103000,Wien Hbf,48.185101,16.377113
1,Vienna,48.2083,16.3725,AT,1973403.0,True,Vienna,8101818,Wien St. Marx,48.187984,16.399550
2,Brussels,50.8467,4.3525,BE,1235192.0,True,Brussels,8800004,Bruxelles Midi,50.835376,4.335694
3,Brussels,50.8467,4.3525,BE,1235192.0,True,Brussels,8800002,Bruxelles-Nord,50.860239,4.361454
4,Brussels,50.8467,4.3525,BE,1235192.0,True,Brussels,8800003,Bruxelles-Central,50.845492,4.357064
...,...,...,...,...,...,...,...,...,...,...,...
102,Ljubljana,46.0514,14.5061,SI,284293.0,True,Ljubljana,7900141,Ljubljana Litostroj,46.077196,14.489906
103,Ljubljana,46.0514,14.5061,SI,284293.0,True,Ljubljana,7900266,Ljubljana Stegne,46.086817,14.479965
104,Bratislava,48.1439,17.1097,SK,475503.0,True,Bratislava,5600207,Bratislava hl.st.,48.158910,17.106466
105,Bratislava,48.1439,17.1097,SK,475503.0,True,Bratislava,5600582,Bratislava-Petrzalka,48.120679,17.100448


# Rertieve Timetables for existing city stations

currently timetables for DB offers limited data. Other timetables are required to be used

In [122]:
header_timetables = {
    "DB-Client-Id": DB_CLIENT_ID,
    "DB-Api-Key": DB_API_KEY,
    "accept": "application/xml"
}


def fetch_hourly_plan(eva_no, date_str, hour_str):
    """
    Fetches the planned timetable for a specific station, date (YYMMDD), and hour (HH).
    """
    url = f"{TIMETABLES_V1_URL}/plan/{eva_no}/{date_str}/{hour_str}"
    success = False
    rertries = 0
    plan_hour = ""

    
    while not success and rertries < 3:
        response = requests.get(url, headers=header_timetables)
        status_code  = response.status_code

        if status_code == 200:
            success = True
            retries = 0
            plan_hour =  response.text
        elif status_code == 404 or status_code == 401:
            print("Access Error: Check API")
            break 
        elif status_code == 429:
            time.sleep(1)
        retries += 1
            
    return plan_hour

def parse_hour_plan(hour_xml:str):
    pass

# test run 
fetch_hourly_plan("8103000","251229","12")


'<?xml version=\'1.0\' encoding=\'UTF-8\'?><timetable station=\'Wien Hbf\'><s id="-641886370829105986-2512291130-3"><tl f="F" t="p" o="81" c="EC" n="204"/><ar pt="2512291158" pp="12A-B" fb="EC 204" ppth="Wien Westbahnhof|Wien Meidling"/><dp pt="2512291210" pp="12A-B" fb="EC 204" pde="Krakow Glowny" ppth="Breclav|Hodonin|Stare Mesto u Uherského Hradiste|Otrokovice|Prerov|Hranice na Morave|Ostrava-Svinov|Ostrava hl.n.|Bohumin"/></s><s id="-721807130937592434-2512291213-1"><tl f="F" t="p" o="81" c="ICE" n="90"/><dp pt="2512291213" pp="8A-B" fb="ICE 90" ppth="Wien Meidling|St.Pölten Hbf|Linz Hbf|Passau Hbf|Plattling|Regensburg Hbf|Nürnberg Hbf|Coburg|Erfurt Hbf|Leipzig Hbf|Lutherstadt Wittenberg Hbf|Berlin Südkreuz|Berlin Hbf|Berlin Gesundbrunnen"/></s><s id="-6130228582536441801-2512290618-10"><tl f="F" t="p" o="51" c="EC" n="203"/><ar pt="2512291149" pp="6A-B" fb="EC 203" pde="Krakow Glowny" ppth="Bohumin|Ostrava hl.n.|Ostrava-Svinov|Hranice na Morave|Prerov|Otrokovice|Stare Mesto u Uher

Using v6.db.transport.rest

In [25]:
from typing import Optional, List, Dict, Any

BASE_URL = "https://v6.db.transport.rest"
RATE_LIMIT_SLEEP = 1.0  # Seconds to sleep between requests to respect 100 req/min
MAX_RETRIES = 3

def fetch_route_raw(origin_id: str, dest_id: str, departure_dt: datetime) -> Optional[Dict]:
    """
    Helper 1: Performs the actual API request for a specific date/time.
    Handles rate limiting and retries.
    """
    params = {
        "from": origin_id,
        "to": dest_id,
        "departure": departure_dt.isoformat(),
        "results": 1,           # We just need the fastest connection for this slot
        "national": "true",     # Prefer long-distance
        "nationalExpress": "true",
        # "profile": "dbweb"      # Web profile often has better international data
    }

    attempts = 0
    while attempts < MAX_RETRIES:
        try:
            response = requests.get(f"{BASE_URL}/journeys", params=params, timeout=10)
            
            if response.status_code == 200:
                time.sleep(RATE_LIMIT_SLEEP) # Polite wait
                return response.json()
            
            elif response.status_code == 429:
                wait = 2 * (attempts + 1)
                print(f"    ⚠️ Rate limit (429). Sleeping {wait}s...")
                time.sleep(wait)
                attempts += 1
            else:
                print(f"    ❌ Error {response.status_code}")
                return None
                
        except Exception as e:
            print(f"    ❌ Request failed: {e}")
            attempts += 1
            time.sleep(1)
            
    return None

route_warsaw_berlin  = fetch_route_raw("5100065", "8011160", datetime.now())

In [ ]:
def extract_journey_metrics(journey_data: Dict) -> Optional[Dict]:
    """
    Helper 2: Parses raw JSON to extract duration, transfers, and train names.
    """
    journeys = journey_data.get('journeys', [])
    if not journeys:
        return None

    # We asked for 1 result, so take the first
    best_journey = journeys[0]
    legs = best_journey.get('legs', [])
    
    if not legs:
        return None

    # Calculate timestamps and duration
    dep_str = legs[0]['departure']
    arr_str = legs[-1]['arrival']
    t_dep = datetime.fromisoformat(dep_str)
    t_arr = datetime.fromisoformat(arr_str)
    duration_minutes = (t_arr - t_dep).total_seconds() / 60

    # Extract train names for context (e.g., "ICE 100")
    trains = [
        leg.get('line', {}).get('name', '') 
        for leg in legs 
        if leg.get('mode') == 'train'
    ]

    return {
        "departure_time": dep_str,
        "arrival_time": arr_str,
        "duration_minutes": int(duration_minutes),
        "transfers": len(legs) - 1,
        "trains": ", ".join(filter(None, trains)),
        "origin_name": legs[0]['origin']['name'],
        "destination_name": legs[-1]['destination']['name']
    }

extract_journey_metrics(route_warsaw_berlin)



{'departure_time': '2025-12-30T22:16:00+01:00',
 'arrival_time': '2025-12-31T06:14:00+01:00',
 'duration_minutes': 478,
 'transfers': 0,
 'trains': '',
 'origin_name': 'Warszawa Centralna',
 'destination_name': 'Berlin Hbf'}

In [28]:
def get_best_weekly_connection(origin_id: str, dest_id: str, start_date: datetime) -> Optional[Dict]:
    """
    Helper 3: Loops through 7 days to find the minimum duration for a pair.
    """
    best_metrics = None
    min_duration = float('inf')

    # Check the same time for the next 7 days (e.g., every morning at 08:00)
    # This accounts for weekends vs weekdays schedules
    for day_offset in range(7):
        current_date = start_date + timedelta(days=day_offset)
        print(f"  Checking {current_date.strftime('%Y-%m-%d')}...", end="\r")
        
        raw_data = fetch_route_raw(origin_id, dest_id, current_date)
        
        if raw_data:
            metrics = extract_journey_metrics(raw_data)
            if metrics:
                # Update if this day offers a faster trip
                if metrics['duration_minutes'] < min_duration:
                    min_duration = metrics['duration_minutes']
                    best_metrics = metrics
                    # Add IDs back to the dict for the final dataframe
                    best_metrics['origin_id'] = origin_id
                    best_metrics['destination_id'] = dest_id
                    best_metrics['best_day'] = current_date.strftime('%A') # e.g. "Monday"

    return best_metrics

get_best_weekly_connection("5100065", "8011160", datetime.now() - timedelta(days=7))

{'departure_time': '2025-12-23T22:16:00+01:00',
 'arrival_time': '2025-12-24T06:14:00+01:00',
 'duration_minutes': 478,
 'transfers': 0,
 'trains': '',
 'origin_name': 'Warszawa Centralna',
 'destination_name': 'Berlin Hbf',
 'origin_id': '5100065',
 'destination_id': '8011160',
 'best_day': 'Tuesday'}

In [29]:
from itertools import permutations


def get_next_representative_weekday():
    """
    Finds the next Tuesday or Wednesday. 
    Mid-week days have the most consistent 'standard' schedules.
    """
    now = datetime.now()
    # 0=Mon, 1=Tue, 2=Wed...
    days_ahead = (1 - now.weekday() + 7) % 7 # Target Tuesday
    if days_ahead == 0: days_ahead = 7 # If today is Tuesday, get next week
    
    target_date = now + timedelta(days=days_ahead)
    # Set to 08:00 AM - Peak morning traffic usually has the best connections
    return target_date.replace(hour=8, minute=0, second=0, microsecond=0)

def fetch_fastest_connection(origin_id: str, dest_id: str) -> Optional[Dict]:
    """
    Makes a SINGLE smart request to find the fastest connection.
    Retrieves 5 options and picks the minimum duration.
    """
    target_date = get_next_representative_weekday()
    
    params = {
        "from": origin_id,
        "to": dest_id,
        "departure": target_date.isoformat(),
        "results": 5,           # Get 5 options in ONE request
        "national": "true",     # Prefer High Speed
        "nationalExpress": "true",
        "transfers": 4          # Allow complex routes if they are faster
    }

    # Retry logic for stability
    for attempt in range(3):
        try:
            response = requests.get(f"{BASE_URL}/journeys", params=params, timeout=10)
            
            if response.status_code == 200:
                data = response.json()
                journeys = data.get('journeys', [])
                
                if not journeys:
                    return None

                # Optimization: Process all 5 results locally to find the minimum
                min_duration = float('inf')
                best_journey = None

                for journey in journeys:
                    if not journey.get('legs'): continue
                    
                    dep = datetime.fromisoformat(journey['legs'][0]['departure'])
                    arr = datetime.fromisoformat(journey['legs'][-1]['arrival'])
                    duration = (arr - dep).total_seconds() / 60
                    
                    if duration < min_duration:
                        min_duration = duration
                        best_journey = journey
                        best_journey['calculated_duration'] = int(duration)

                # Extract details from the winner
                legs = best_journey['legs']
                train_names = [l.get('line', {}).get('name', '') for l in legs if l.get('mode') == 'train']

                time.sleep(RATE_LIMIT_SLEEP) # Respect limits
                return {
                    "origin_id": origin_id,
                    "destination_id": dest_id,
                    "origin_name": legs[0]['origin']['name'],
                    "destination_name": legs[-1]['destination']['name'],
                    "min_duration_minutes": best_journey['calculated_duration'],
                    "transfers": len(legs) - 1,
                    "trains": ", ".join(filter(None, train_names)),
                    "check_date": target_date.strftime("%Y-%m-%d")
                }

            elif response.status_code == 429:
                time.sleep(3 * (attempt + 1)) # Backoff
            elif response.status_code >= 500:
                time.sleep(2)
            else:
                return None
                
        except Exception as e:
            time.sleep(1)

    return None

def get_efficient_network_data(eva_ids: List[str]) -> pd.DataFrame:
    """
    Main Runner.
    Reduced Complexity: O(N * (N-1)) requests instead of O(N * (N-1) * 7).
    """
    # Use permutations because A->B might be slightly different than B->A 
    # (e.g. connections matching up)
    pairs = list(permutations(eva_ids, 2))
    
    print(f"🚀 Optimized Search: {len(eva_ids)} stations -> {len(pairs)} routes.")
    print(f"📅 Using representative weekday: {get_next_representative_weekday().strftime('%A, %Y-%m-%d')}")
    
    results = []
    
    for i, (origin, dest) in enumerate(pairs):
        print(f"[{i+1}/{len(pairs)}] {origin} -> {dest}...", end=" ", flush=True)
        
        data = fetch_fastest_connection(origin, dest)
        
        if data:
            print(f"✅ {data['min_duration_minutes']} min")
            results.append(data)
        else:
            print(f"❌ No route")

    return pd.DataFrame(results)

In [ ]:
routes_data = get_efficient_network_data(citiies_stations['eva_id'].tolist())
routes_data

🚀 Optimized Search: 43 stations -> 1806 routes.
📅 Using representative weekday: Tuesday, 2026-01-06
[1/1806] 8103000 -> 8800004... ✅ 622 min
[2/1806] 8103000 -> 8800007... ✅ 653 min
[3/1806] 8103000 -> 5400014... ✅ 253 min
[4/1806] 8103000 -> 8011160... ✅ 429 min
[5/1806] 8103000 -> 8000096... ✅ 391 min
[6/1806] 8103000 -> 8000261... ✅ 253 min
[7/1806] 8103000 -> 8002549... ✅ 524 min
[8/1806] 8103000 -> 8003368... ✅ 481 min
[9/1806] 8103000 -> 8000105... ✅ 384 min
[10/1806] 8103000 -> 8000085... ✅ 499 min
[11/1806] 8103000 -> 8010205... ✅ 378 min
[12/1806] 8103000 -> 8000080... ✅ 546 min
[13/1806] 8103000 -> 8000098... ✅ 525 min
[14/1806] 8103000 -> 8000050... ✅ 514 min
[15/1806] 8103000 -> 8010085... ✅ 400 min
[16/1806] 8103000 -> 8000152... ✅ 439 min
[17/1806] 8103000 -> 8000284... ✅ 252 min
[18/1806] 8103000 -> 8000086... ✅ 512 min
[19/1806] 8103000 -> 8601242... ✅ 910 min
[20/1806] 8103000 -> 7100064... ✅ 1195 min
[21/1806] 8103000 -> 8700012... ✅ 714 min
[22/1806] 8103000 -> 87000